# 名詞限定クラスタ0：ランダム30発言の内容確認

2016〜2026年の名詞限定クラスタリング結果から、クラスタ0を固定シードで30件抽出します。発言本文を省略せず表示し、ノイズやトピックの一貫性を目視確認します。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'analysis').exists():
    ROOT = ROOT.parent
INPUT_PATH = ROOT / 'analysis/clustering_2016_2026_nouns/speech_clusters.csv'
OUTPUT_PATH = ROOT / 'analysis/clustering_2016_2026_nouns/cluster_0_random_30.csv'
CLUSTER_ID = 0
SAMPLE_SIZE = 30
RANDOM_STATE = 42

## 1. クラスタ0の母集団を確認

In [ ]:
df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig', parse_dates=['date'])
required = {'speechID', 'date', 'year', 'speaker', 'speakerPosition', 'speech', 'speech_cleaned', 'review_flags', 'speechURL', 'cluster', 'cluster_label', 'centroid_distance', 'representativeness'}
missing = required - set(df.columns)
assert not missing, f'必要な列がありません: {sorted(missing)}'
assert df['speechID'].is_unique
cluster_population = df.loc[df['cluster'].eq(CLUSTER_ID)].copy()
assert len(cluster_population) >= SAMPLE_SIZE
display(pd.Series({
    '全クラスタリング発言': len(df),
    'クラスタ0発言': len(cluster_population),
    'クラスタ0ラベル': cluster_population['cluster_label'].iat[0],
    '抽出件数': SAMPLE_SIZE,
    '乱数シード': RANDOM_STATE,
}).to_frame('value'))
display(cluster_population.groupby('year', as_index=False).agg(speeches=('speechID', 'size'), speakers=('speaker', 'nunique')))

## 2. 固定シードで30件を抽出

In [ ]:
sample = cluster_population.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).copy()
sample.insert(0, 'sample_order', range(1, SAMPLE_SIZE + 1))
sample.loc[:, 'review_decision'] = ''
sample.loc[:, 'review_notes'] = ''
assert len(sample) == SAMPLE_SIZE
assert sample['speechID'].is_unique
assert sample['cluster'].eq(CLUSTER_ID).all()
display(sample[['sample_order', 'date', 'year', 'speaker', 'speakerPosition', 'review_flags', 'centroid_distance', 'speechID']])

## 3. 発言内容を目視確認

`speech_cleaned` はクラスタリング対象本文、`speech` は確認用の原文です。

In [ ]:
for row in sample.itertuples(index=False):
    position = row.speakerPosition if pd.notna(row.speakerPosition) and str(row.speakerPosition).strip() else '役職なし'
    flags = row.review_flags if pd.notna(row.review_flags) and str(row.review_flags).strip() else 'なし'
    cleaned = str(row.speech_cleaned).replace('\n', '  \n')
    original = str(row.speech).replace('\n', '  \n')
    display(Markdown(
        f"### {row.sample_order}. {row.speaker}（{position}）— {row.date:%Y-%m-%d}\n"
        f"- 発言ID: `{row.speechID}`\n"
        f"- レビューフラグ: {flags}\n"
        f"- 重心距離: {row.centroid_distance:.6f}\n"
        f"- [国会会議録の発言ページ]({row.speechURL})\n\n"
        f"**クリーニング本文**\n\n{cleaned}\n\n"
        f"<details><summary>原文を表示</summary>\n\n{original}\n\n</details>"
    ))

## 4. レビュー用CSVを保存

`review_decision` と `review_notes` は人手確認用の空欄です。Excelでも開けるUTF-8 BOM付きで保存します。

In [ ]:
export_columns = [
    'sample_order', 'speechID', 'date', 'year', 'speaker', 'speakerPosition',
    'speech', 'speech_cleaned', 'review_flags', 'speechURL', 'cluster',
    'cluster_label', 'centroid_distance', 'representativeness',
    'review_decision', 'review_notes',
]
sample.loc[:, export_columns].to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
saved = pd.read_csv(OUTPUT_PATH, encoding='utf-8-sig')
assert len(saved) == SAMPLE_SIZE
assert saved['speechID'].tolist() == sample['speechID'].tolist()
print(f'保存完了: {OUTPUT_PATH}')